# ML-07 — Baseline Action Score and Top-10 Review

**Lane locked: Lane 2 — Refresh / Content Opportunity Scoring.** I am keeping the lane from Weeks 1–3. The customer is an editor choosing which visible pages to inspect for a possible content refresh; inspection can also lead to monitoring or a technical investigation.

Run all cells with Python 3 and `pandas`, `numpy`, `duckdb`, `huggingface_hub`, and `scikit-learn`. The notebook is standalone: it reads the pinned March warehouse partition, reuses its ignored cache, and regenerates `work/outputs/baseline_action_score.csv` and the committed aggregate receipt `work/outputs/w04_baseline_metrics.json`. Authentication uses a saved HF login, `HF_TOKEN`, or a Colab Secret; never put a token in a cell. The existing local task login also works by launching Jupyter with `HF_HOME` pointing to the ignored `work/outputs/hf_cache` directory.

Guidance: [baseline skill](../../skills/building-baselines/SKILL.md), [data skill](../../skills/flyrank/flyrank-data/SKILL.md), and [Week 3 contract](w03_data_contract.ipynb). The CSV and raw data stay out of Git. Only a small pseudonymized top-ten preview and aggregate outputs are displayed.

**Temporal contract:** March 1–14 inputs → March 15–16 buffer → March 17 decision → March 17–30 outcome. Source grain is one client–page–day; score grain is one client–page at that decision. Only TRUE GSC availability and valid counts qualify. A page needs all 14 past days and at least 100 past impressions to enter the queue. Missing future coverage excludes it from retrospective metrics, **not** the queue. June remains sealed. The reporting buffer does not establish actual historic ingestion or revision timestamps.

## 1. Two signal checks, then one rule

**Initial rule idea:** Prioritize high-exposure pages, adding urgency for unusually variable daily impressions. Before deciding whether to keep that volatility bonus, inspect two fixed bucket tables on development clients only. Each table shows page count **n**, number of clients, observed decline count, decline rate, and median past exposure.

- **Volume hypothesis:** More past impressions indicate more exposure worth protecting and might identify more future-decline cases. Volume is the signal linked to the real **quick-win flag example named in the assignment**. This is a link to its input concept, not a recreation of FlyRank's proprietary thresholds or a claim that these pages are quick wins.
- **Volatility hypothesis:** A larger daily-impression coefficient of variation (CV = daily population SD / daily mean) might indicate greater future-decline risk. This is my candidate signal, not a claimed FlyRank flag.

Use the same 25% client holdout and seed 42 as Week 3. Its results were already inspected in Week 3, so it is a **reused comparison set**, not an untouched final test. Signal auditing and rule selection use the other clients. The label is the same >20% decline in the later equal-length window. Labels appear only in audits/evaluation, never score inputs. Bucket associations are directional, not causal; large clients and within-client dependence can distort pooled rates.

In [1]:
from pathlib import Path
import os
import platform
import importlib.metadata
import pandas as pd
import numpy as np
import duckdb
from huggingface_hub import HfApi, hf_hub_download, get_token
from IPython.display import display, Markdown

SEED = 42
K = 20
REPO_ID = "FlyRank/internship-warehouse"
REVISION = "50cbf7c3909d07be4d1b5906b4d09e882e5acbf2"
PARTITION = "fact_content_daily_performance/month=2026-03/data_0.parquet"
MIN_PAST_IMPRESSIONS = 100
# Find the repository from either its root or work/notebooks; Colab uses /content.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "skills/README.md").is_file()), Path.cwd())
CACHE = ROOT / "work/outputs/hf_cache"
CACHE.mkdir(parents=True, exist_ok=True)
print("Pinned warehouse revision:", REVISION)
print("Partition:", PARTITION)
print("Python:", platform.python_version())
print({p: importlib.metadata.version(p) for p in ["pandas", "duckdb", "huggingface_hub", "scikit-learn"]})

Pinned warehouse revision: 50cbf7c3909d07be4d1b5906b4d09e882e5acbf2
Partition: fact_content_daily_performance/month=2026-03/data_0.parquet
Python: 3.14.6
{'pandas': '3.0.6', 'duckdb': '1.5.5', 'huggingface_hub': '2.0.0', 'scikit-learn': '1.9.1'}


/var/home/tanzimul/Repos/github.com/tanzimul3islam/flyrank-ml-internship-starter/work/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Retrieve credentials without printing, embedding, or saving them in notebook outputs.
token = os.environ.get("HF_TOKEN") or get_token()
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None

# A pinned local cache is reusable without re-downloading or contacting the service.
try:
    source_path = hf_hub_download(REPO_ID, PARTITION, repo_type="dataset",
                                  revision=REVISION, cache_dir=CACHE, local_files_only=True)
except Exception:
    if not token:
        raise RuntimeError("Warehouse access is required: accept the gate and use hf auth login, "
                           "an HF_TOKEN environment variable, or a Colab HF_TOKEN Secret. "
                           "Do not paste a token into this notebook or chat.") from None
    try:
        source_path = hf_hub_download(REPO_ID, PARTITION, repo_type="dataset",
                                      revision=REVISION, cache_dir=CACHE, token=token)
    except Exception:
        raise RuntimeError("Could not read the gated March partition. Check access approval, "
                           "READ-token permission, and network connectivity; then rerun.") from None
finally:
    token = None

con = duckdb.connect()
con.execute("SET threads = 2")
con.execute("SET memory_limit = '2GB'")
# Relational projection prevents accidental reading/display of private or unused fields.
SOURCE_FIELDS = ["report_date", "client_hash_id", "content_hash_id", "gsc_data_available",
                 "ga4_data_available", "gsc_impressions", "gsc_clicks", "gsc_avg_position"]
relation = con.read_parquet(source_path)
missing = set(SOURCE_FIELDS) - set(relation.columns)
assert not missing, f"Warehouse schema changed; missing expected fields: {sorted(missing)}"
relation.project(", ".join(SOURCE_FIELDS)).create_view("march")
print("March partition loaded. Source fields verified; no final-month table was opened.")

March partition loaded. Source fields verified; no final-month table was opened.


In [3]:
# Recheck source grain before aggregation; display only a count of violations.
violations = con.sql("""
    SELECT COUNT(*) AS bad_keys FROM (
      SELECT report_date, client_hash_id, content_hash_id FROM march
      GROUP BY report_date, client_hash_id, content_hash_id
      HAVING COUNT(*) > 1 OR report_date IS NULL
         OR client_hash_id IS NULL OR content_hash_id IS NULL
    )
""").fetchone()[0]
assert violations == 0, "Invalid page-day grain; do not aggregate."
print("Duplicate/null page-day keys:", violations)
BUILD_SQL = "\nWITH valid_days AS (\n    SELECT *, report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14' AS is_past,\n              report_date BETWEEN DATE '2026-03-17' AND DATE '2026-03-30' AS is_future\n    FROM march\n    WHERE gsc_data_available IS TRUE\n      AND gsc_impressions IS NOT NULL AND gsc_impressions >= 0\n      AND gsc_clicks IS NOT NULL AND gsc_clicks >= 0\n), aggregated AS (\n    SELECT client_hash_id, content_hash_id,\n           COUNT(*) FILTER (WHERE is_past) AS past_valid_days,\n           COUNT(*) FILTER (WHERE is_future) AS future_valid_days,\n           MIN(report_date) FILTER (WHERE is_past) AS past_first,\n           MAX(report_date) FILTER (WHERE is_past) AS past_last,\n           MIN(report_date) FILTER (WHERE is_future) AS future_first,\n           MAX(report_date) FILTER (WHERE is_future) AS future_last,\n           SUM(gsc_impressions) FILTER (WHERE is_past) AS past_impressions,\n           SUM(gsc_clicks) FILTER (WHERE is_past) AS past_clicks,\n           SUM(gsc_avg_position * gsc_impressions)\n               FILTER (WHERE is_past AND gsc_avg_position > 0) AS weighted_position_sum,\n           SUM(gsc_impressions)\n               FILTER (WHERE is_past AND gsc_avg_position > 0) AS position_impressions,\n           COUNT(*) FILTER (WHERE is_past AND gsc_impressions > 0) AS past_active_days,\n           STDDEV_POP(gsc_impressions) FILTER (WHERE is_past) AS past_impression_sd,\n           SUM(gsc_impressions) FILTER (WHERE is_future) AS future_impressions\n    FROM valid_days GROUP BY client_hash_id, content_hash_id\n)\nSELECT * FROM aggregated\n"
context = con.sql(BUILD_SQL).df().sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)
past_ok = context.past_valid_days.eq(14) & context.past_impressions.ge(MIN_PAST_IMPRESSIONS)
# This frame deliberately omits every outcome-window column.
past = context.loc[past_ok, ["client_hash_id", "content_hash_id", "past_impressions",
    "past_clicks", "past_valid_days", "past_first", "past_last", "past_active_days",
    "weighted_position_sum", "position_impressions", "past_impression_sd"]].copy()
past["ctr_pct"] = 100 * past.past_clicks / past.past_impressions
past["position"] = past.weighted_position_sum / past.position_impressions.replace(0, np.nan)
past["daily_impression_cv"] = past.past_impression_sd / (past.past_impressions / 14)
assert not past.duplicated(["client_hash_id", "content_hash_id"]).any()
assert pd.to_datetime(past.past_last).max() < pd.Timestamp("2026-03-17")

# Outcome-only table: cannot be passed to the scoring function below.
labeled = context.loc[past_ok & context.future_valid_days.eq(14)].copy().reset_index(drop=True)
labeled["declined"] = (labeled.future_impressions < .8 * labeled.past_impressions).astype("int8")
from sklearn.model_selection import GroupShuffleSplit
train, test = next(GroupShuffleSplit(n_splits=1, test_size=.25, random_state=SEED)
                   .split(labeled, groups=labeled.client_hash_id))
dev_clients = set(labeled.iloc[train].client_hash_id)
holdout_clients = set(labeled.iloc[test].client_hash_id)
assert dev_clients.isdisjoint(holdout_clients)
assert len(labeled) == 57624, "Slice no longer matches the Week 3 contract."
labels = labeled[["client_hash_id", "content_hash_id", "declined"]]
audit = past.merge(labels, on=["client_hash_id", "content_hash_id"], validate="one_to_one")
audit = audit.loc[audit.client_hash_id.isin(dev_clients)].copy()
assert len(audit) == len(train)
print(f"Queue-eligible pages: {len(past):,}; labeled pages: {len(labeled):,}.")
print(f"Audit: {len(audit):,} pages / {len(dev_clients)} clients; comparison: {len(test):,} pages / {len(holdout_clients)} clients.")

def bucket_table(column, edges, names):
    grouped = audit.assign(bucket=pd.cut(audit[column], bins=edges, right=False, labels=names))
    result = grouped.groupby("bucket", observed=False).agg(
        n=("declined", "size"), clients=("client_hash_id", "nunique"),
        declines=("declined", "sum"), decline_rate=("declined", "mean"),
        median_past_impressions=("past_impressions", "median"))
    assert int(result.n.sum()) == len(audit), "Buckets must cover all development pages."
    return result

volume_buckets = bucket_table("past_impressions", [100, 300, 1000, 3000, 10000, np.inf],
                             ["100–299", "300–999", "1,000–2,999", "3,000–9,999", "10,000+"])
volatility_buckets = bucket_table("daily_impression_cv", [0, .25, .5, 1, 2, np.inf],
                                 ["<0.25", "0.25–<0.5", "0.5–<1", "1–<2", "2+"])
print("Signal 1: past volume (quick-win input concept)")
display(volume_buckets)
print("Signal 2: daily impression volatility")
display(volatility_buckets)

Duplicate/null page-day keys: 0


Queue-eligible pages: 60,534; labeled pages: 57,624.
Audit: 14,137 pages / 24 clients; comparison: 43,487 pages / 8 clients.
Signal 1: past volume (quick-win input concept)


,n,clients,declines,decline_rate,median_past_impressions
bucket,,,,,
100–299,4879,24,1381,0.283050,176.0
300–999,5355,20,1638,0.305882,571.0
"1,000–2,999",2583,18,848,0.328300,1579.0
"3,000–9,999",1084,15,361,0.333026,4538.5
"10,000+",236,12,77,0.326271,14211.0


Signal 2: daily impression volatility


,n,clients,declines,decline_rate,median_past_impressions
bucket,,,,,
<0.25,1736,18,469,0.270161,1143.0
0.25–<0.5,7280,23,2341,0.321566,434.0
0.5–<1,4733,22,1341,0.283330,431.0
1–<2,382,17,151,0.395288,475.5
2+,6,3,3,0.500000,1039.5


**Volume verdict: MIXED.** Decline rates rise from **28.3% (n=4,879)** at 100–299 impressions to **33.3% (n=1,084)** at 3,000–9,999, then soften to **32.6% (n=236)** at 10,000+. This is a modest pooled association, not a strict dose-response or proof of refresh value. Volume remains useful as an **exposure-priority policy**, but I will not turn it into a claimed calibrated risk score.

**Volatility verdict: MIXED.** Rates are **27.0%, 32.2%, 28.3%, 39.5%, 50.0%**, with **n=1,736 / 7,280 / 4,733 / 382 / 6** respectively. The middle buckets contradict a simple increasing-risk story, and the highest bucket is tiny. The proposed volatility bonus is removed; its value remains review context only. This negative check prevents a noisy signal from receiving extra score weight.

**Final rule, frozen as `volume_first_v1`:** Among pages with 14 valid past days and at least 100 past impressions, inspect the pages with the most past impressions first. The score is simply the past impression count; every row receives the single reason code **`visible_exposure_priority`** and action label **`inspect_for_refresh`**. Inspection is mandatory before editing: high exposure alone does not establish deterioration, outdated content or refresh benefit.

There are no fitted weights, threshold searches, volatility bonus, product flags, future counts or target-derived inputs. The 100-impression eligibility floor is inherited from Week 3. Ties are resolved by pseudonymous client/page IDs solely for deterministic ordering. Freeze this rule before Week 5; a failed comparison should be reported rather than repaired using the comparison labels.

## 2. Build the ranked queue and write the CSV

The queue includes **all decision-time eligible pages**, including those with unknown future outcomes. The three rule-output fields are `score`, one `reason_code`, and one `action_label`. Other exported measurements are past-window review context, not extra score terms. IDs support routing/grouping, never learned prediction.

After writing the queue, evaluate on the exact **43,487 labeled comparison pages / 8 clients** from Week 3. Report decline-proxy precision@20 beside prevalence (random selection's expected precision), and ROC-AUC as a secondary ranking diagnostic. These are not independent editorial judgments. Week 5 must compare its model on these same IDs, proxy, windows and K; the June final test remains unused.

In [4]:
import json
import hashlib
from sklearn.metrics import roc_auc_score

RULE_VERSION = "volume_first_v1"
REASON_CODE = "visible_exposure_priority"
ACTION_LABEL = "inspect_for_refresh"
KEYS = ["client_hash_id", "content_hash_id"]
SCORE_INPUTS = ["past_impressions"]

def score_pages(inputs):
    assert list(inputs.columns) == SCORE_INPUTS
    assert inputs.past_impressions.ge(MIN_PAST_IMPRESSIONS).all()
    return inputs.past_impressions.astype(float)

queue = past[KEYS + ["past_impressions", "ctr_pct", "position", "daily_impression_cv"]].copy()
queue["score"] = score_pages(past[SCORE_INPUTS])
queue["reason_code"] = REASON_CODE
queue["action_label"] = ACTION_LABEL
queue["decision_date"] = "2026-03-17"
queue = queue.sort_values(["score", *KEYS], ascending=[False, True, True]).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
assert queue.score.is_monotonic_decreasing
assert len(queue) == len(past) and not queue.duplicated(KEYS).any()
assert queue.reason_code.nunique() == queue.action_label.nunique() == 1
assert not any("future" in c or "declin" in c or "label_copy" in c for c in queue.columns)
OUT = ROOT / "work/outputs"
OUT.mkdir(parents=True, exist_ok=True)
CSV_PATH = OUT / "baseline_action_score.csv"
queue.to_csv(CSV_PATH, index=False, float_format="%.10g")
print(f"Wrote {len(queue):,} ranked rows to work/outputs/baseline_action_score.csv (untracked).")

# The join happens after scoring/export. Labels cannot change scores or queue eligibility.
evaluation = queue.merge(labels, on=KEYS, validate="one_to_one")
evaluation = evaluation.loc[evaluation.client_hash_id.isin(holdout_clients)].sort_values("rank")
assert len(evaluation) == len(test)
assert set(map(tuple, evaluation[KEYS].to_numpy())) == set(map(tuple, labeled.iloc[test][KEYS].to_numpy()))
assert evaluation.declined.nunique() == 2 and len(evaluation) >= K
precision = float(evaluation.head(K).declined.mean())
base_rate = float(evaluation.declined.mean())
auc = float(roc_auc_score(evaluation.declined, evaluation.score))
metric_table = pd.DataFrame([
    {"measure": f"Rule decline-proxy precision@{K}", "value": precision},
    {"measure": "Random-selection expectation / decline prevalence", "value": base_rate},
    {"measure": "Precision lift above prevalence (absolute)", "value": precision - base_rate},
    {"measure": "Rule ROC-AUC (secondary)", "value": auc},
])
display(metric_table)
print(f"Comparison top-{K}: {int(evaluation.head(K).declined.sum())}/{K} declined; "
      f"{evaluation.head(K).client_hash_id.nunique()} represented clients.")
print("Reused Week 3 comparison set; no new final-test claim.")

# Fingerprints let Week 5 prove its split matches without committing row-level data.
def key_fingerprint(frame):
    rows = frame[KEYS].sort_values(KEYS).itertuples(index=False, name=None)
    text = "\n".join("|".join(map(str, r)) for r in rows)
    return hashlib.sha256(text.encode()).hexdigest()

receipt = {
    "rule_version": RULE_VERSION, "dataset_revision": REVISION, "partition": PARTITION,
    "lane": "Lane 2: Refresh / Content Opportunity Scoring", "lane_locked": True,
    "feature_window": ["2026-03-01", "2026-03-14"], "decision_date": "2026-03-17",
    "outcome_window": ["2026-03-17", "2026-03-30"], "minimum_past_impressions": MIN_PAST_IMPRESSIONS,
    "label_definition": "future_14d_impressions < 0.8 * past_14d_impressions; both windows fully covered",
    "score_inputs": SCORE_INPUTS, "score_formula": "past_impressions",
    "reason_code": REASON_CODE, "action_label": ACTION_LABEL, "random_seed": SEED,
    "queue_rows": len(queue), "labeled_rows": len(labeled), "development_rows": len(audit),
    "comparison_rows": len(evaluation), "comparison_clients": len(holdout_clients),
    "comparison_status": "Previously inspected in Week 3; not an untouched final test",
    "comparison_keys_sha256": key_fingerprint(evaluation),
    "development_keys_sha256": key_fingerprint(labeled.iloc[train]),
    "k": K, "proxy_precision_at_k": precision, "decline_base_rate": base_rate,
    "precision_lift_absolute": precision-base_rate, "roc_auc": auc,
    "top_k_clients": int(evaluation.head(K).client_hash_id.nunique()),
    "signal_verdicts": {"volume": "MIXED", "volatility": "MIXED"},
    "volume_buckets": json.loads(volume_buckets.reset_index().to_json(orient="records")),
    "volatility_buckets": json.loads(volatility_buckets.reset_index().to_json(orient="records")),
    "versions": {p: importlib.metadata.version(p) for p in ["pandas", "duckdb", "huggingface_hub", "scikit-learn"]},
}

Wrote 60,534 ranked rows to work/outputs/baseline_action_score.csv (untracked).


,measure,value
0,Rule decline-proxy precision@20,0.450000
1,Random-selection expectation / decline prevalence,0.374365
2,Precision lift above prevalence (absolute),0.075635
3,Rule ROC-AUC (secondary),0.532410


Comparison top-20: 9/20 declined; 3 represented clients.
Reused Week 3 comparison set; no new final-test claim.


**Measured baseline result:** 9 of the first 20 comparison pages declined: **precision@20 = 45%**, versus **37.44%** prevalence (a **7.56 percentage-point** difference). ROC-AUC is **0.5324**, close to chance. With only 20 evaluated ranking slots and a previously inspected comparison set, this is a modest descriptive benchmark, not reliable evidence of editorial usefulness. The rule is deliberately left unchanged for Week 5.

## 3. Top-10 review — action, reason, confidence and what would make it wrong

These are the **first ten rows of the full decision-time queue**, not the comparison subset or cherry-picked examples. The concise public preview uses pseudonyms and past measurements only. The ten notes below are a skeptical review of the observed metrics; neither the assistant nor the intern has inspected the actual page text, intent or technical condition. Every confidence note concerns actionability, not measurement certainty.

A low CTR is not automatically bad: position, query mix, intent and SERP layout matter. In particular, the seventh pick's positive average position is **below 1**—a data-quality warning that the Week 3 `>0` check did not catch. Position does not affect this volume-only score, but it makes that pick's action case especially weak.

In [5]:
top10 = queue.head(10).copy()
display(top10[["rank", "content_hash_id", "score", "ctr_pct", "position", "daily_impression_cv",
               "reason_code", "action_label"]])
# Individually written after inspecting these ten past-window records, not their outcomes.
review_notes = {
 "content_e8a52cf3d5988c07": ("low", "138,793 impressions put it first; position 16.16 merits an intent/coverage inspection", "the page is current and position reflects competition rather than missing content"),
 "content_36e53e9c707674fc": ("low", "105,148 impressions give it high exposure, but position 33.14 is weak", "authority or a technical issue, rather than content freshness, explains the position"),
 "content_7172a7fad43f0998": ("low", "102,917 impressions put it third although position 2.98 is already strong", "it already satisfies intent and an edit would disrupt a successful page"),
 "content_b99ea6861864dea5": ("low", "86,624 impressions justify inspection; position 4.17 alone does not imply a gap", "0.197% CTR is normal for its SERP layout or query mix"),
 "content_e7b5dd4dff461ad2": ("low", "84,409 impressions give it priority; CV 0.61 is context, not a score bonus", "a short-lived demand spike explains the variation and no update is needed"),
 "content_7c6373141eae744a": ("low", "84,160 impressions with 0.056% CTR at position 5.99 warrants a contextual review", "zero-click intent or search-result layout explains the low CTR"),
 "content_9c057b66c30a3abb": ("very low", "83,770 impressions trigger the rule, but zero clicks and position 0.104 weaken the evidence", "tracking or aggregation errors invalidate the apparent opportunity; verify data first"),
 "content_3df3f32f3fd58dea": ("low", "81,456 impressions keep it high despite position 24.45", "ranking weakness comes from competition or technical constraints that refreshing will not fix"),
 "content_acbcc847f8996314": ("low", "77,413 impressions at position 3.52 mean substantial exposure", "the content is already effective and the low CTR follows query intent rather than a content defect"),
 "content_471d9cabce329a66": ("low", "75,458 impressions and position 4.41 earn a slot, with CV only 0.31", "the page is stable and current, so another client's lower-volume page has a more actionable issue"),
}
assert len(review_notes) == 10 and set(top10.content_hash_id) == set(review_notes), "Top ten changed: rewrite the review."
review_lines=[]
for row in top10.itertuples(index=False):
    confidence, why, wrong = review_notes[row.content_hash_id]
    review_lines.append(f"{row.rank}. `{row.content_hash_id}` — **{row.action_label}**; "
                        f"`{row.reason_code}`: {why}. Action confidence: **{confidence}**. "
                        f"**Wrong if:** {wrong}.")
display(Markdown("\n\n".join(review_lines)))
print(f"Top ten span {top10.client_hash_id.nunique()} clients; "
      f"largest client's share: {top10.client_hash_id.value_counts().max()}/10.")

,rank,content_hash_id,score,ctr_pct,position,daily_impression_cv,reason_code,action_label
0,1,content_e8a52cf3d5988c07,138793.0,0.244969,16.161262,0.339589,visible_exposure_priority,inspect_for_refresh
1,2,content_36e53e9c707674fc,105148.0,0.104614,33.144111,0.187829,visible_exposure_priority,inspect_for_refresh
2,3,content_7172a7fad43f0998,102917.0,0.444047,2.983006,0.424666,visible_exposure_priority,inspect_for_refresh
3,4,content_b99ea6861864dea5,86624.0,0.197405,4.166467,0.364185,visible_exposure_priority,inspect_for_refresh
4,5,content_e7b5dd4dff461ad2,84409.0,1.057944,4.351550,0.606612,visible_exposure_priority,inspect_for_refresh
5,6,content_7c6373141eae744a,84160.0,0.055846,5.990078,0.566356,visible_exposure_priority,inspect_for_refresh
6,7,content_9c057b66c30a3abb,83770.0,0.000000,0.104130,1.876951,visible_exposure_priority,inspect_for_refresh
7,8,content_3df3f32f3fd58dea,81456.0,0.143636,24.454871,0.411636,visible_exposure_priority,inspect_for_refresh
8,9,content_acbcc847f8996314,77413.0,0.155013,3.519887,0.364584,visible_exposure_priority,inspect_for_refresh
9,10,content_471d9cabce329a66,75458.0,0.255771,4.405391,0.314427,visible_exposure_priority,inspect_for_refresh


1. `content_e8a52cf3d5988c07` — **inspect_for_refresh**; `visible_exposure_priority`: 138,793 impressions put it first; position 16.16 merits an intent/coverage inspection. Action confidence: **low**. **Wrong if:** the page is current and position reflects competition rather than missing content.

2. `content_36e53e9c707674fc` — **inspect_for_refresh**; `visible_exposure_priority`: 105,148 impressions give it high exposure, but position 33.14 is weak. Action confidence: **low**. **Wrong if:** authority or a technical issue, rather than content freshness, explains the position.

3. `content_7172a7fad43f0998` — **inspect_for_refresh**; `visible_exposure_priority`: 102,917 impressions put it third although position 2.98 is already strong. Action confidence: **low**. **Wrong if:** it already satisfies intent and an edit would disrupt a successful page.

4. `content_b99ea6861864dea5` — **inspect_for_refresh**; `visible_exposure_priority`: 86,624 impressions justify inspection; position 4.17 alone does not imply a gap. Action confidence: **low**. **Wrong if:** 0.197% CTR is normal for its SERP layout or query mix.

5. `content_e7b5dd4dff461ad2` — **inspect_for_refresh**; `visible_exposure_priority`: 84,409 impressions give it priority; CV 0.61 is context, not a score bonus. Action confidence: **low**. **Wrong if:** a short-lived demand spike explains the variation and no update is needed.

6. `content_7c6373141eae744a` — **inspect_for_refresh**; `visible_exposure_priority`: 84,160 impressions with 0.056% CTR at position 5.99 warrants a contextual review. Action confidence: **low**. **Wrong if:** zero-click intent or search-result layout explains the low CTR.

7. `content_9c057b66c30a3abb` — **inspect_for_refresh**; `visible_exposure_priority`: 83,770 impressions trigger the rule, but zero clicks and position 0.104 weaken the evidence. Action confidence: **very low**. **Wrong if:** tracking or aggregation errors invalidate the apparent opportunity; verify data first.

8. `content_3df3f32f3fd58dea` — **inspect_for_refresh**; `visible_exposure_priority`: 81,456 impressions keep it high despite position 24.45. Action confidence: **low**. **Wrong if:** ranking weakness comes from competition or technical constraints that refreshing will not fix.

9. `content_acbcc847f8996314` — **inspect_for_refresh**; `visible_exposure_priority`: 77,413 impressions at position 3.52 mean substantial exposure. Action confidence: **low**. **Wrong if:** the content is already effective and the low CTR follows query intent rather than a content defect.

10. `content_471d9cabce329a66` — **inspect_for_refresh**; `visible_exposure_priority`: 75,458 impressions and position 4.41 earn a slot, with CV only 0.31. Action confidence: **low**. **Wrong if:** the page is stable and current, so another client's lower-volume page has a more actionable issue.

Top ten span 4 clients; largest client's share: 4/10.


## 4. Weak picks and leakage check

**Weak picks:** #7 is a data-verification priority, not a justified refresh: the sub-one position and zero clicks undermine the opportunity interpretation. #3 and #9 already have strong positions, so high volume may waste review slots on successful pages. #2 and #8 may have authority or technical problems rather than stale content. These are exactly the errors a volume-only baseline can make. Keep them visible; removing them after seeing the top ten would silently change the frozen rule.

**Concentration:** Four of the top ten belong to one client. A global exposure queue can neglect smaller clients; do not silently introduce client quotas during model comparison. A future operational policy could evaluate quotas separately with the editor.

**Leakage boundary:** The score function receives only `past_impressions`. Both audited signals and all exported review context come from March 1–14; IDs only break ties. The March 17–30 target is used for development signal tables and a post-scoring comparison join. Future availability never determines queue membership. Product flags, staleness snapshot metadata, fixed-window query summaries and label-derived columns are absent from the score.

**Limits:** Report dates do not prove when data arrived; the two-day buffer remains an assumption. The >20% decline label is an observed traffic proxy, not editorial actionability or refresh impact. The same clients were evaluated in Week 3, and this one-month comparison is not a fresh final test. Verdicts are pooled associations without causal or significance claims.

**Week 5 handoff:** Keep `volume_first_v1`, eligibility, outcome definition, client split and K=20 fixed. Use the committed JSON key fingerprints to verify an identical comparison pool. Compare the learned ranking with this baseline on the same IDs, and report failures honestly; independent editorial judgments are still required to claim useful refresh recommendations.

In [6]:
# Check the artifacts and final invariants, without loading any additional window.
written = pd.read_csv(CSV_PATH)
assert len(written) == len(queue) and list(written.columns) == list(queue.columns)
assert written.reason_code.eq(REASON_CODE).all() and written.action_label.eq(ACTION_LABEL).all()
assert np.array_equal(written.score.to_numpy(), queue.score.to_numpy())
assert set(SCORE_INPUTS) == {"past_impressions"}
assert not any(c in queue.columns for c in ["declined", "future_impressions", "future_valid_days", "deliberate_label_copy"])
assert int((context.loc[past_ok, "future_valid_days"] != 14).sum()) == len(queue) - len(labeled)
assert set(queue.content_hash_id) == set(past.content_hash_id)
receipt["unlabeled_queue_rows"] = len(queue)-len(labeled)
receipt["top10_review_count"] = len(review_notes)
receipt["top10_clients"] = int(top10.client_hash_id.nunique())
receipt["top10_largest_client_count"] = int(top10.client_hash_id.value_counts().max())
receipt["queue_csv_sha256"] = hashlib.sha256(CSV_PATH.read_bytes()).hexdigest()
receipt["checks"] = {"source_grain_passed": True, "future_columns_absent_from_queue": True,
                     "score_inputs_past_only": True, "csv_roundtrip_passed": True}
METRICS_PATH = OUT / "w04_baseline_metrics.json"
METRICS_PATH.write_text(json.dumps(receipt, indent=2, allow_nan=False) + "\n")
print(f"Queue retains {len(queue)-len(labeled):,} pages with unknown outcomes; these are not counted as negatives.")
print("Saved aggregate receipt: work/outputs/w04_baseline_metrics.json")
print("CSV regenerated and checked. Rule frozen; only notebook and aggregate JSON belong in this commit.")

Queue retains 2,910 pages with unknown outcomes; these are not counted as negatives.
Saved aggregate receipt: work/outputs/w04_baseline_metrics.json
CSV regenerated and checked. Rule frozen; only notebook and aggregate JSON belong in this commit.


## 5. Self-check

- [x] Lane 2 confirmed and locked for the remaining weeks.
- [x] Two executed signal bucket tables show n and client counts; each has the verdict MIXED.
- [x] Volume is explicitly linked to the session's quick-win input concept; no proprietary flag is reconstructed.
- [x] One transparent, frozen rule produces a score, one reason code, and one action label.
- [x] The notebook writes the full decision-time queue to `work/outputs/baseline_action_score.csv`.
- [x] Exactly ten top-ranked rows have individual action, reason, confidence and wrong-if notes; weak picks are named.
- [x] Score inputs are past-only; no future coverage, product flag or label enters the rule.
- [x] Proxy precision@20 is reported beside prevalence on the Week 3 comparison pool.
- [x] Aggregate metrics, slice definition and split fingerprints are saved as the JSON receipt.
- [x] All cells executed top to bottom and the final outputs inspected.

**Submission:** Commit this executed notebook and the aggregate JSON; keep CSV/Parquet/cache files ignored. Push the commit and submit the repo URL on the assignment card.

**AI assistance:** An assistant drafted the code and the ten metric-based review notes using the requested repo skills. This is not a human page-content review, a stakeholder-confirmed capacity decision, or a causal refresh experiment.